This notebook add Cybernetic Reaction Sandbox to ATS-flow model xml files

- scenario1: constant transport BC for NH4+, NO3-, DOC
- scenario2: dynamic transport BC for NH4+, NO3- from ELM; constant transport BC for DOC
- scenario3: dynamic transport BC for NH4+, NO3-, DOC from ELM

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
from pathlib import Path
import shutil
import subprocess
import h5py as h5
import matplotlib.pyplot as plt
import re

In [ ]:
# Parameters cell
import json
with open('config.json', 'r') as f:
    config = json.load(f)
watershed_name = config['watershed_name']
hucs           = [config['hucs']]
site_name      = config['site_name']

# # simulation control
# start_year_spinup         = config['start_year_spinup']
# end_year_spinup           = config['end_year_spinup']
# nyears_steadystate_spinup = config['nyears_steadystate_spinup']
# nyears_cyclic_spinup      = config['nyears_cyclic_spinup']
# start_year_transient      = config['start_year_transient']
# end_year_transient        = config['end_year_transient']

flag_scenario = 's1' # choose which scenario
outputs = {}

In [ ]:
# Add atspflotranutils to sys.path
base_dir = Path().resolve().parent  # Assumes the notebook is inside the notebooks/ directory
utils_path = base_dir / 'atspflotranutils'
sys.path.append(str(utils_path))

# Check if the path was added successfully
print("Paths in sys.path:")
print("\n".join(sys.path))

# pflotranate atsflow xml to atspflotran xml

In [ ]:
from pflotranate_2d import pflotranate
# from pflotranate_3d import pflotranate
# from pflotranate_2dtracers import pflotranate
pflotranate_path = base_dir / 'atspflotranutils' / 'pflotranate_2d'

import importlib
importlib.reload(pflotranate)

In [ ]:
# Prepare '../casecybernetic-run{1,2}' folders
source_path_run1 = base_dir / 'caseflow-run1' / f'{site_name}_nx100_nz18.run1.v1.5.xml'
target_folder_run1 = base_dir / f'casecybernetic-run1.{flag_scenario}'
target_path_run1 = target_folder_run1 / f'{site_name}_nx100_nz18.run1.v1.5.xml'
os.makedirs(target_folder_run1, exist_ok=True)
shutil.copy(source_path_run1, target_path_run1)

source_path_run2 = base_dir / 'caseflow-run2' / f'{site_name}_nx100_nz18.run2.v1.5.xml'
target_folder_run2 = base_dir / f'casecybernetic-run2.{flag_scenario}'
target_path_run2 = target_folder_run2 / f'{site_name}_nx100_nz18.run2.v1.5.xml'
os.makedirs(target_folder_run2, exist_ok=True)
shutil.copy(source_path_run2, target_path_run2)

## copy corresponding folder-reactions to case folders
source_path_reactions = pflotranate_path / 'reactions'
try:
    shutil.copytree(source_path_reactions, target_folder_run1 / 'reactions', dirs_exist_ok=True)
    shutil.copytree(source_path_reactions, target_folder_run2 / 'reactions', dirs_exist_ok=True)
except FileExistsError as e:
    print(f"Target folder already exists: {e}")

In [ ]:
# Simulate command-line arguments
command = f"python {pflotranate_path / 'pflotranate.py'} {target_path_run1}"

try:
    subprocess.run(command, shell=True, check=True)
    #print(f"Successfully executed: {command}")
except subprocess.CalledProcessError as e:
    print(f"Error occurred: {e}")

# modify the xml file for casecybernetic-run1

In [ ]:
# amanzi_xml, included in AMANZI_SRC_DIR/tools/amanzi_xml
import amanzi_xml.utils.io as aio
import amanzi_xml.utils.search as asearch
import amanzi_xml.utils.errors as aerrors
from amanzi_xml.common.parameter import Parameter
from amanzi_xml.common.parameter_list import ParameterList

In [ ]:
xml_filename = base_dir / f'casecybernetic-run1.{flag_scenario}' / 'NF01_nx100_nz18.run1.v1.5_pflotran.xml'
xml = aio.fromFile(xml_filename)

## revise the initial conditions for flow part

In [ ]:
caseflow_run1_checkpoint = "../../caseflow-run1/NF01/checkpoint_final.h5"

subsurface_flow_IC = asearch.find_path(xml, ['PKs', 'subsurface flow', 'initial condition', 'restart file'])
subsurface_flow_IC.set("value", caseflow_run1_checkpoint)
print(subsurface_flow_IC)
aio.toFile(xml, xml_filename)

## revise the soil domain info used for DOC injection source terms

In [ ]:
#soil_domain_region = "{NRCS-68932, NRCS-68901}" # copied from last section in 1-main_workflow_OakCreek.NF01.ats1.5.ipynb
outputs['soil_region_string'] = f'../data-processed/{site_name}/soil_region.txt'
with open(outputs['soil_region_string'], 'r') as f:
    loaded_string = f.read().strip()
soil_domain_region = f"{{{loaded_string}}}"

subsurface_mass_source = asearch.find_path(xml, ['PKs', 'source terms', 'component mass source', 'DOC production', 'regions'])
subsurface_mass_source.set("value", soil_domain_region)
print(subsurface_mass_source)
aio.toFile(xml, xml_filename)

## revise the DOC injection hdf5 file info

In [ ]:
elm_filename = "../../data-processed/NF01/NF01_DOC_source_spinup_10yr_2011_2015_fdom001.h5"

subsurface_mass_source = asearch.find_path(xml, ['PKs', 'source terms', 'component mass source', 'DOC production', 'source function', 'file'])
subsurface_mass_source.set("value", elm_filename)
print(subsurface_mass_source)
aio.toFile(xml, xml_filename)

## revise other configurations specifically for casecybernetic

In [ ]:
## cycle driver
cycle_driver = asearch.find_path(xml, ['cycle driver', 'end time'])
cycle_driver.set("value", "365")
aio.toFile(xml, xml_filename)

cycle_driver = asearch.find_path(xml, ['cycle driver', 'end cycle'])
cycle_driver.set("value", "10000000") # 100,000 -> 166.4day, so for 3650day, change to 1e7
aio.toFile(xml, xml_filename)

In [ ]:
## visualization
visualization = asearch.find_path(xml, ['visualization', 'domain', 'times start period stop'])
visualization.set("value", "{ 0,0.1,-1}")
aio.toFile(xml, xml_filename)

visualization = asearch.find_path(xml, ['visualization', 'domain', 'time units'])
visualization.set("value", "s")
aio.toFile(xml, xml_filename)

visualization = asearch.find_path(xml, ['visualization', 'surface', 'times start period stop'])
visualization.set("value", "{ 0,0.1,-1}")
aio.toFile(xml, xml_filename)

visualization = asearch.find_path(xml, ['visualization', 'surface', 'time units'])
visualization.set("value", "s")
aio.toFile(xml, xml_filename)

## revise initial conditions set in file pflotran_chemistry_cybernetic_smoothstep_pyc.txt
- this is only for the cybernetic spinup run1

In [ ]:
## load dynamic boundary conditions for NH4+, NO3-, and DOC from ELMimport h5py
hdf5_path = "../data-processed/NF01/NF01_CNbc_conc_spinup_10yr_2011_2015_fdom001_k15.h5"
with h5.File(hdf5_path, "r") as f:
    time_spinup_bc = f['Time'][:]
    nh4_conc_spinup_bc = f['NH4+ mol water basis [molS molH^-1]'][:]
    no3_conc_spinup_bc = f['NO3- mol water basis [molS molH^-1]'][:]
    doc_conc_spinup_bc = f['DOC mol water basis v1 [molS molH^-1]'][:]

In [ ]:
## plot and calculate averaged values
fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axs[0].plot(time_spinup_bc, nh4_conc_spinup_bc, color='b')
axs[0].set_ylabel('NH4+ [molS/molH]')
axs[0].set_title('NH4+ Concentration Over Time')

axs[1].plot(time_spinup_bc, no3_conc_spinup_bc, color='g')
axs[1].set_ylabel('NO3- [molS/molH]')
axs[1].set_title('NO3- Concentration Over Time')

axs[2].plot(time_spinup_bc, doc_conc_spinup_bc, color='r')
axs[2].set_ylabel('DOC [molS/molH]')
axs[2].set_title('DOC Concentration Over Time')
axs[2].set_xlabel('Time [days]')

plt.tight_layout()
plt.show()

In [ ]:
## calculate the average values
avg_nh4 = nh4_conc_spinup_bc.mean()
avg_no3 = no3_conc_spinup_bc.mean()
avg_doc = doc_conc_spinup_bc.mean()

## noticing that the concentration unit in pflotran_chemistry_cybernetic_smoothstep_pyc.txt is in molS/L
## so we need to convert the unit from molS/molH to molS/L
rho_m = 55000. # molH/m3, water molar density
avg_nh4_molS_L = avg_nh4 * rho_m / 1000  # convert from molS/molH to molS/L
avg_no3_molS_L = avg_no3 * rho_m / 1000  # convert from molS/molH to molS/L
avg_doc_molS_L = avg_doc * rho_m / 1000  # convert from molS/molH to molS/L

print(f"Time-averaged NH4+ concentration: {avg_nh4_molS_L:.3e} [molS/L]")
print(f"Time-averaged NO3- concentration: {avg_no3_molS_L:.3e} [molS/L]")
print(f"Time-averaged DOC concentration: {avg_doc_molS_L:.3e} [molS/L]")

## manual config
avg_doc_molS_L = 2.0e-4

In [ ]:
## revise file pflotran_chemistry_cybernetic_smoothstep_pyc.txt
## for cybernetic spinup run1
def replace_species(text, constraint_name, replacements):
    pattern = re.compile(
        rf"(CONSTRAINT {constraint_name}\s+CONCENTRATIONS\s+)(.*?)(/)",
        re.DOTALL
    )
    def repl(match):
        block = match.group(2)
        for species, value in replacements.items():
            block = re.sub(
                rf"({re.escape(species)}\s+)[\deE\.\-]+",
                lambda m: m.group(1) + f"{value:.5e}",
                block
            )
        return match.group(1) + block + match.group(3)
    return pattern.sub(repl, text)

file_path = target_folder_run1 /'reactions/pflotran_chemistry_cybernetic_smoothstep_pyc.txt'

replacements = {
    "CH2O(aq)": avg_doc_molS_L,
    "NH4+": avg_nh4_molS_L,
    "NO3-": avg_no3_molS_L
}

with open(file_path, "r") as f:
    content = f.read()

content = replace_species(content, "ICsubsurface", replacements)
content = replace_species(content, "ICsurface", replacements)

with open(file_path, "w") as f:
    f.write(content)


# modify the xml file for casecybernetic-run2

## revise the initial condition for flow part

## revise the soil domain info used for DOC injection source terms

## revise the DOC injection hdf5 file info

## revise other configurations specifically for casecybernetic

## revise initial conditions set in file pflotran_chemistry_cybernetic_smoothstep_pyc.txt

# xml_input_convert to xmls for phong's branch

In [ ]:
# convert xml files to phong's branch
xmlinputconvert_path = Path('/home/xiao284/softwares/ats-15-25sep-phong/repos/amanzi/src/physics/ats/tools/input_converters')

# ## caseflow-run0
# tmp_xml = base_dir / 'caseflow-run0' / 'NF01_nx100_nz18.run0.v1.5.xml'
# tmp_xml_o = base_dir / 'caseflow-run0' / 'NF01_nx100_nz18.run0.vphong.xml'
# command = f"python {xmlinputconvert_path / 'xml-1.5-master.py'} {tmp_xml} -o {tmp_xml_o}"
# try:
#     subprocess.run(command, shell=True, check=True)
# except subprocess.CalledProcessError as e:
#     print(f"Error occurred: {e}")

# ## caseflow-run1
# tmp_xml = base_dir / 'caseflow-run1' / 'NF01_nx100_nz18.run1.v1.5.xml'
# tmp_xml_o = base_dir / 'caseflow-run1' / 'NF01_nx100_nz18.run1.vphong.xml'
# command = f"python {xmlinputconvert_path / 'xml-1.5-master.py'} {tmp_xml} -o {tmp_xml_o}"
# try:
#     subprocess.run(command, shell=True, check=True)
# except subprocess.CalledProcessError as e:
#     print(f"Error occurred: {e}")

## casecybernetic-run1
tmp_xml = base_dir / f'casecybernetic-run1.{flag_scenario}' / 'NF01_nx100_nz18.run1.v1.5_pflotran.xml'
tmp_xml_o = base_dir / f'casecybernetic-run1.{flag_scenario}' / 'NF01_nx100_nz18.run1.vphong_pflotran.xml'
command = f"python {xmlinputconvert_path / 'xml-1.5-master.py'} {tmp_xml} -o {tmp_xml_o}"
try:
    subprocess.run(command, shell=True, check=True)
except subprocess.CalledProcessError as e:
    print(f"Error occurred: {e}")

# ## casecybernetic-run2
# tmp_xml = base_dir / 'casecybernetic-run1' / 'NF01_nx100_nz18.run1.v1.5_pflotran.xml'
# tmp_xml_o = base_dir / 'casecybernetic-run1' / 'NF01_nx100_nz18.run1.vphong_pflotran.xml'
# command = f"python {xmlinputconvert_path / 'xml-1.5-master.py'} {tmp_xml} '-o' {tmp_xml_o}"
# try:
#     subprocess.run(command, shell=True, check=True)
# except subprocess.CalledProcessError as e:
#     print(f"Error occurred: {e}")